# A/B-тест геймификации на образовательной платформе

*Кейс «Оценка результатов A/B-тестирования продуктовой фичи»*

## Цель проекта
Оценить успешность проведённого A/B-теста по внедрению новой механики геймификации — системы достижений за регулярное решение задач — на образовательной платформе. На основе статистического анализа необходимо принять аргументированное бизнес-решение: раскатить новую функцию на всех пользователей, отправить её на доработку или отказаться от неё.

## Описание проекта
Продуктовая команда EdTech-компании выдвинула гипотезу: если добавить в личный кабинет ученика элементы геймификации, то конверсия в полное прохождение курса (Completion Rate) и метрики удержания (Retention 7 и 14 дня) статистически значимо вырастут.

Для проверки гипотезы был запущен A/B-тест. Пользователи были случайным образом разделены на две группы: группа A — контрольная, со старым интерфейсом; группа B — тестовая, с интерфейсом, в который добавлена геймификация. Эксперимент длился 3 недели.

## Данные

Датасет состоит из двух файлов:

**`ab_test_users_groups.csv`**: пользователи эксперимента.

| Колонка | Описание |
|---|---|
| `user_id` | идентификатор пользователя |
| `group` | группа эксперимента |
| `signup_date` | дата регистрации на платформе |
| `grade` | класс ученика |

**`ab_test_events.csv`**: лог действий пользователей.

| Колонка | Описание |
|---|---|
| `user_id` | идентификатор пользователя |
| `timestamp` | дата и время события |
| `event_type` | тип события |

Типы событий:
- `login` — вход на платформу
- `homework_started` — начало выполнения домашнего задания
- `homework_completed` — домашнее задание выполнено
- `return_visit_d7` — возврат на платформу на 7-й день
- `return_visit_d14` — возврат на платформу на 14-й день

## Гипотезы

**Completion Rate**

**H0:** Completion Rate в группах A и B равны – $CR_A = CR_B$

**H1:** Completion Rate в группах A и B не равны – $CR_A \neq CR_B$

**Retention D7**

**H0:** Retention 7-го дня в группах A и B равны – $R^{D7}_A = R^{D7}_B$

**H1:** Retention 7-го дня в группах A и B не равны – $R^{D7}_A \neq R^{D7}_B$

**Retention D14**

**H0:** Retention 14-го дня в группах A и B равны – $R^{D14}_A = R^{D14}_B$

**H1:** Retention 14-го дня в группах A и B не равны – $R^{D14}_A \neq R^{D14}_B$

## Импорт библиотек

In [75]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

## План анализа

1. Подготовка данных
2. Sanity check
3. Расчёт метрик (+ Time to Value)
4. EDA
5. Выбор критерия
6. Статистический расчёт (+ MDE, bootstrap)
7. Интерпретация результатов (+ сегментация по классам)
8. Вывод

## Подготовка данных

### 1.1 Загрузка и первичный осмотр данных

In [76]:
events = pd.read_csv("../data/raw/ab_test_events.csv")
users = pd.read_csv("../data/raw/ab_test_users_groups.csv")

In [77]:
events.info()
events.head()

<class 'pandas.DataFrame'>
RangeIndex: 179330 entries, 0 to 179329
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   user_id     179330 non-null  str  
 1   timestamp   179330 non-null  str  
 2   event_type  179330 non-null  str  
dtypes: str(3)
memory usage: 4.1 MB


,user_id,timestamp,event_type
0,U300281,2026-05-04 00:06:44,homework_completed
1,U300733,2026-05-04 00:19:56,homework_completed
2,U304499,2026-05-04 00:22:07,login
3,U304266,2026-05-04 00:23:42,homework_started
4,U304266,2026-05-04 00:23:42,homework_started


In [78]:
users.info()
users.head()

<class 'pandas.DataFrame'>
RangeIndex: 6008 entries, 0 to 6007
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      6008 non-null   str  
 1   group        6008 non-null   str  
 2   signup_date  6008 non-null   str  
 3   grade        6008 non-null   int64
dtypes: int64(1), str(3)
memory usage: 187.9 KB


,user_id,group,signup_date,grade
0,U300000,B,2026-03-09,5
1,U300001,B,2025-09-07,9
2,U300002,B,2026-01-11,9
3,U300003,A,2025-06-07,5
4,U300004,B,2025-12-25,11


### 1.2 Приведение типов и дубликаты

При осмотре видно, что `timestamp` и `signup_date` имеют строковый тип данных. Для корректной работы с датами приведем их к типу datetime. В `events` заметны две одинаковые строки подряд, поэтому также проверим таблицы на дубликаты.

In [79]:
events["timestamp"] = pd.to_datetime(events["timestamp"])
users["signup_date"] = pd.to_datetime(users["signup_date"])

In [80]:
events.duplicated().sum(), users.duplicated().sum()

(np.int64(2), np.int64(0))

In [81]:
events, users = events.drop_duplicates(), users.drop_duplicates()

### 1.3 Пользователи в обеих группах

Пользователь мог попасть в таблицу дважды с разными данными. Проверим количество уникальных `user_id` и сравним его с числом строк.

In [82]:
events.groupby("user_id").size().describe()

count    6000.000000
mean       29.888000
std        33.994888
min         6.000000
25%        24.000000
50%        28.000000
75%        33.000000
max       842.000000
dtype: float64

In [83]:
users.nunique()

user_id        6000
group             2
signup_date     370
grade             7
dtype: int64

In [84]:
users[users["user_id"].duplicated(keep=False)].sort_values("user_id")

,user_id,group,signup_date,grade
357,U300357,A,2026-03-19,9
6000,U300357,B,2026-03-19,9
460,U300460,B,2025-08-12,6
6001,U300460,A,2025-08-12,6
920,U300920,B,2025-04-09,10
6002,U300920,A,2025-04-09,10
1014,U301014,A,2026-03-28,10
6003,U301014,B,2026-03-28,10
1087,U301087,B,2025-06-27,10
6004,U301087,A,2025-06-27,10


8 пользователей числятся одновременно в обеих группах. Определить, какой интерфейс видел такой пользователь, – невозможно – удаляем целиком из обеих таблиц, – на результаты это не повлияет.

In [85]:
duplicates = users[users.duplicated(subset="user_id", keep=False)]["user_id"].unique()

users = users[~users["user_id"].isin(duplicates)]
events = events[~events["user_id"].isin(duplicates)]

### 1.4 Боты
В прошлом шаге при изучении таблицы `events` мы выяснили – у типичного ученика около 28 событий, у 75% – не больше 33, а максимальное значение – 842 – аномально большое число действий. Найдем всех таких пользователей.

In [87]:
events_per_user = events.groupby("user_id").size().sort_values(ascending=False)
events_per_user.head(20)

user_id
U303729    842
U304167    829
U305378    793
U302840    790
U304266    780
U300573    771
U300733    770
U302379    765
U300281    760
U300158    738
U304499    729
U303438    711
U300628     52
U301601     51
U303387     51
U300180     50
U300042     49
U305026     49
U300388     49
U302161     49
dtype: int64

In [88]:
bots = events_per_user[events_per_user > 500]

users = users[~users["user_id"].isin(bots.index)]
events = events[~events["user_id"].isin(bots.index)]

Обнаружено 12 пользоватлей с аномальным количеством действий (711+); следующий по активности идет пользователь с 52 действиями – разрыв более чем в 10 раз. Удаляем их из обеих таблиц.

### 1.5 Объединение таблиц

Для расчёта метрик каждому событию нужны группа и класс пользователя. Объединим `users` и `events` по `user_id`.

In [89]:
events_full = users.merge(events, on="user_id", how="left")
events_full

,user_id,group,signup_date,grade,timestamp,event_type
0,U300000,B,2026-03-09,5,2026-05-04 19:36:00,login
1,U300000,B,2026-03-09,5,2026-05-07 19:35:00,login
2,U300000,B,2026-03-09,5,2026-05-07 19:42:00,homework_started
3,U300000,B,2026-03-09,5,2026-05-08 17:05:00,login
4,U300000,B,2026-03-09,5,2026-05-08 17:11:00,homework_started
...,...,...,...,...,...,...
169828,U305999,A,2025-06-19,11,2026-05-20 19:56:00,homework_started
169829,U305999,A,2025-06-19,11,2026-05-20 20:16:00,homework_completed
169830,U305999,A,2025-06-19,11,2026-05-21 20:47:00,login
169831,U305999,A,2025-06-19,11,2026-05-21 20:53:00,homework_started


In [90]:
events["event_type"].value_counts()

event_type
login                 72304
homework_started      65108
homework_completed    25971
return_visit_d7        4505
return_visit_d14       1945
Name: count, dtype: int64